# AI STOCK PREDICTION SYSTEM
### Indian Market Data & Intelligent Stock Analytics
### Data Collection, Preprocessing & Exploratory Data Analysis

**AI & Financial Data Analytics**  
**Market:** Indian Stock Market (NSE)  
**Currency:** Indian Rupee (₹ INR)  
**Default Stock:** Reliance Industries (`RELIANCE.NS`)

---
> **Objective:** Collect, clean, preprocess, scale, analyze, and prepare real-world Indian stock market data for future Machine Learning model development. No prediction models are trained at this stage.

## 1. Import Libraries
We import standard Python libraries: `numpy`, `pandas`, `yfinance`, `matplotlib.pyplot`, and `sklearn` preprocessing tools.

In [ ]:
# Import essential data science and financial libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Scikit-learn preprocessing modules
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# Plotting configurations
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("All libraries imported successfully!")

## 2. Data Collection
Fetch real-world historical data for **Reliance Industries (RELIANCE.NS)** from the National Stock Exchange of India (NSE) via Yahoo Finance.

In [ ]:
# Fetch 1 year of historical data for Reliance Industries from Yahoo Finance
ticker = "RELIANCE.NS"
data = yf.download(ticker, period="1y", progress=False)

# If yfinance returns multi-index columns, flatten them
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Reset index so Date becomes a standard column
data = data.reset_index()
data['Date'] = pd.to_datetime(data['Date']).dt.strftime('%Y-%m-%d')

print(f"Data collected for {ticker}:")
print(f"Total records: {len(data)}")
data.head()

## 3. Dataset Description
Understanding the columns in our stock dataset:
- **Date**: Trading date (YYYY-MM-DD)
- **Open**: Opening stock price in ₹ INR
- **High**: Highest price reached during the session in ₹ INR
- **Low**: Lowest price reached during the session in ₹ INR
- **Close**: Closing stock price at session conclusion in ₹ INR
- **Volume**: Total quantity of shares transacted during the session

In [ ]:
# Summary statistics of numerical columns (prices in ₹ INR)
print("Summary Statistics (Prices in ₹ INR):")
data[['Open', 'High', 'Low', 'Close', 'Volume']].describe().round(2)

## 4. Dataset Information
Examine column data types, non-null counts, and memory footprint.

In [ ]:
# Display dataset schema and structural info
print("Dataset Information:")
data.info()
print("\nShape of dataset:", data.shape)

## 5. Data Cleaning
Standardize formats, ensure chronological sorting, cast numeric columns, and eliminate redundant fields.

In [ ]:
# Record counts before cleaning
print("BEFORE CLEANING:")
print("Rows:", len(data))
print("Duplicates:", data.duplicated().sum())
print("Missing Values:", data.isnull().sum().sum())

# Ensure sorted chronologically by Date
data = data.sort_values(by='Date').reset_index(drop=True)

# Ensure numeric columns are rounded to 2 decimals
for col in ['Open', 'High', 'Low', 'Close']:
    data[col] = data[col].round(2)

print("\nAFTER CLEANING:")
print("Rows:", len(data))
print("Duplicates:", data.duplicated().sum())
print("Missing Values:", data.isnull().sum().sum())

## 6. Missing Value Analysis
Check for null or missing values across all columns and apply Forward Fill (`ffill`) followed by Backward Fill (`bfill`).

In [ ]:
# Audit missing values per column
print("Missing values per column:")
print(data.isnull().sum())

# Apply Forward Fill (last valid price is propagated forward)
data = data.ffill().bfill()

print("\nMissing values after treatment:")
print(data.isnull().sum())

## 7. Duplicate Detection
Identify and purge any duplicate rows.

In [ ]:
# Check for duplicated rows
num_duplicates = data.duplicated().sum()
print("Duplicate rows found:", num_duplicates)

# Remove duplicates if any exist
data = data.drop_duplicates()
print("Dataset rows after dropping duplicates:", len(data))

## 8. Data Transformation
Derive fundamental financial measures: High-Low Spread and Intraday Price Change.

In [ ]:
# Transform raw prices into informative derived metrics
data['Price_Spread'] = (data['High'] - data['Low']).round(2)
data['Price_Change'] = (data['Close'] - data['Open']).round(2)

print("Transformed features added:")
data[['Date', 'Open', 'Close', 'Price_Spread', 'Price_Change']].head()

## 9. Data Reduction
Select high-signal attributes and optimize memory consumption.

In [ ]:
# Measure initial memory footprint
initial_mem = data.memory_usage(deep=True).sum() / 1024

# Keep only relevant core features
selected_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Price_Change', 'Price_Spread']
data = data[[c for c in selected_cols if c in data.columns]]

final_mem = data.memory_usage(deep=True).sum() / 1024
print(f"Initial memory: {initial_mem:.2f} KB | Reduced memory: {final_mem:.2f} KB")

## 10. Categorical Encoding
Demonstrate Label Encoding and One-Hot Encoding for categorical attributes (e.g., Stock Tickers or Sectors).

> **Note:** Continuous price columns (Open, High, Low, Close, Volume) are already numerical and must NEVER be encoded.

In [ ]:
# Create demonstration categorical DataFrame
stocks_demo = pd.DataFrame({
    "Stock": ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "SBIN.NS"],
    "Sector": ["Energy", "IT", "IT", "Banking", "Banking"]
})

# 1. Label Encoding
le = LabelEncoder()
stocks_demo['Sector_Label'] = le.fit_transform(stocks_demo['Sector'])

# 2. One-Hot Encoding
ohe = pd.get_dummies(stocks_demo['Sector'], prefix='Sector', dtype=int)
demo_result = pd.concat([stocks_demo, ohe], axis=1)

print("Categorical Encoding Demonstration:")
demo_result

## 11. Feature Scaling
Why scale? Financial features like Volume (millions of shares) and Price (thousands of ₹) have vastly different scales. Scaling ensures fair gradient updates in future ML models.

## 12. Normalization (MinMaxScaler)
Rescales features into a bounded range of `[0, 1]`.

In [ ]:
# MinMaxScaler Demonstration
scale_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
minmax = MinMaxScaler()
normalized_array = minmax.fit_transform(data[scale_cols])
df_norm = pd.DataFrame(normalized_array, columns=[c + '_Norm' for c in scale_cols])

print("MinMax Normalized Data Sample (Range: [0, 1]):")
df_norm.head().round(4)

## 13. Standardization (StandardScaler)
Rescales features so that `mean = 0` and `standard deviation = 1`.

In [ ]:
# StandardScaler Demonstration
standard = StandardScaler()
standardized_array = standard.fit_transform(data[scale_cols])
df_std = pd.DataFrame(standardized_array, columns=[c + '_Std' for c in scale_cols])

print("Standardized Data Sample (Mean=0, Std=1):")
df_std.head().round(4)

## 14. Exploratory Data Analysis (EDA)
Compute descriptive distributions and correlations.

In [ ]:
# Correlation matrix across numeric columns
corr = data[['Open', 'High', 'Low', 'Close', 'Volume']].corr().round(3)
print("Correlation Matrix:")
print(corr)

## 15. Data Visualization
Create required visualizations using Matplotlib:
1. Closing Price vs Date (₹ INR)
2. Opening Price vs Date (₹ INR)
3. Trading Volume vs Date
4. Open vs Close Scatter Plot
5. Correlation Heatmap

In [ ]:
# 1. Closing Price vs Date
plt.figure(figsize=(12, 5))
plt.plot(data['Date'], data['Close'], color='#06b6d4', linewidth=2, label='Close Price (₹)')
plt.title(f'{ticker} - Closing Price vs Date', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.xticks(data['Date'][::30], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
# Observation: The closing price fluctuates over time, showing changes in the stock's market valuation.

In [ ]:
# 2. Trading Volume vs Date
plt.figure(figsize=(12, 4))
plt.bar(data['Date'], data['Volume'], color='#8b5cf6', alpha=0.7, label='Volume')
plt.title(f'{ticker} - Trading Volume vs Date', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Shares Traded')
plt.xticks(data['Date'][::30], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
# Observation: High volume spikes indicate strong institutional market participation or news announcements.

In [ ]:
# 3. Open vs Close Scatter Analysis
plt.figure(figsize=(8, 6))
plt.scatter(data['Open'], data['Close'], color='#10b981', alpha=0.6)
plt.title('Open Price vs Close Price (₹)', fontsize=14, fontweight='bold')
plt.xlabel('Open Price (₹)')
plt.ylabel('Close Price (₹)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
# Observation: Strong positive linear relationship confirming that opening price strongly anchors closing price.

## 16. Feature Engineering
Engineer domain-specific financial features:
- **SMA_20**: 20-day Simple Moving Average
- **EMA_20**: 20-day Exponential Moving Average
- **Daily_Return**: Daily percentage price change
- **Volume_Change**: Day-to-day percentage volume variation
- **RSI**: 14-period Relative Strength Index

In [ ]:
# Calculate technical indicators
data['SMA_20'] = data['Close'].rolling(window=20).mean().round(2)
data['EMA_20'] = data['Close'].ewm(span=20, adjust=False).mean().round(2)
data['Daily_Return'] = data['Close'].pct_change().round(4)
data['Volume_Change'] = data['Volume'].pct_change().round(4)

# Calculate 14-period RSI
delta = data['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss.replace(0, np.nan))
data['RSI'] = (100 - (100 / (1 + rs))).round(2)

# Backward fill rolling indicator NaNs
data = data.bfill()

print("Engineered Features Created:")
data[['Date', 'Close', 'SMA_20', 'EMA_20', 'Daily_Return', 'RSI']].tail()

## 17. Feature Selection
Select relevant input attributes ($X$) and target variable ($y$).
- **Input Features ($X$)**: `Open`, `High`, `Low`, `Volume`, `SMA_20`, `EMA_20`, `Daily_Return`, `Volume_Change`, `RSI`
- **Target ($y$)**: `Next_Day_Close` (Next trading day's closing price in ₹ INR)

In [ ]:
# Create target variable: Next Day Close Price
data['Next_Day_Close'] = data['Close'].shift(-1)

# Drop the final row where target is unknown
df_ml = data.dropna(subset=['Next_Day_Close']).copy()

feature_cols = ['Open', 'High', 'Low', 'Volume', 'SMA_20', 'EMA_20', 'Daily_Return', 'Volume_Change', 'RSI']
print("Selected Input Features (X):", feature_cols)
print("Selected Target Variable (y): Next_Day_Close (₹)")

## 18. Feature Extraction
Demonstrate Principal Component Analysis (PCA) to extract low-dimensional representations.

In [ ]:
# Standardize features before applying PCA
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(df_ml[feature_cols])

# Extract 2 Principal Components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained Variance by Component: {pca.explained_variance_ratio_ * 100} %")
print(f"Total Variance Captured: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

## 19. Define X and y
Separate the features matrix ($X$) from the target vector ($y$).

In [ ]:
X = df_ml[feature_cols]
y = df_ml['Next_Day_Close']

print("X Shape (Input Features Matrix):", X.shape)
print("y Shape (Target Vector):", y.shape)

## 20. Train-Test Split
Prepare training (80%) and testing (20%) datasets using a chronological time-series split.

> **Crucial Rule:** We do NOT shuffle time-series data to avoid lookahead bias and data leakage.

In [ ]:
# 80% Training, 20% Testing chronological split
train_size = int(len(df_ml) * 0.80)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Training Records (80%): {len(X_train)}")
print(f"Testing Records (20%): {len(X_test)}")
print(f"Training Date Range: {df_ml['Date'].iloc[0]} to {df_ml['Date'].iloc[train_size-1]}")
print(f"Testing Date Range: {df_ml['Date'].iloc[train_size]} to {df_ml['Date'].iloc[-1]}")

## 21. Final Dataset
Export and inspect the final clean, feature-engineered dataset ready for future ML model development.

In [ ]:
# Display first 10 rows of final ML-ready dataset
print("Final Clean Preprocessed Dataset:")
print(f"Dimensions: {df_ml.shape[0]} rows × {df_ml.shape[1]} columns")
df_ml.head(10)

## 22. Conclusion
### Project Implementation Summary
In this phase, we completed:
1. **Data Collection**: Downloaded real NSE stock prices directly in ₹ INR.
2. **Data Cleaning**: Removed duplicates and sorted chronologically.
3. **Missing Value Handling**: Audited nulls and applied Forward Fill.
4. **Data Transformation & Reduction**: Derived price spreads and optimized memory.
5. **Categorical Encoding**: Demonstrated Label and One-Hot encoding.
6. **Feature Scaling**: Implemented Normalization (`MinMaxScaler`) and Standardization (`StandardScaler`).
7. **EDA & Visualizations**: Generated price trends, volume, scatter, and correlation charts.
8. **Feature Engineering**: Created SMA, EMA, returns, and RSI indicators.
9. **Feature Selection & Extraction**: Selected input features, defined target variable in ₹, and extracted PCA components.
10. **Train-Test Preparation**: Created an 80/20 chronological time-series split.

> **Future Scope:** Machine learning model training (LSTM, Random Forest, XGBoost), price predictions, buy/sell recommendations, and portfolio analytics will be developed in future project phases.